# Apresentação

Notebook que eu vou utilizar pra criar todo o código. Aqui eu vou escrever o código e vou depois passar pros scritps.

In [2]:
# Dependencies
import os
import re
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import urljoin
import polars as pl
from tqdm import tqdm
import io
import zipfile

In [3]:
load_dotenv()

True

# Aquisitando dados
Primeiro passo é importar os dados. Eu quero importar de forma organizada, baseado na organização do site. Vou criar uma pasta e salvo os arquivos dentro dela. Mas eu preciso conseguir mapear o site o verificar adequadamente as pastas.

**NOTA:** O "2016 a 2018", se eu for puxar, eu preciso puxar de forma diferente, porque ele está fora de padrão.

In [4]:
url = os.getenv("DATA_RESOUCES")
resp = requests.get(url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

In [5]:
soup

<!DOCTYPE html>

<html lang="pt-br" xml:lang="pt-br" xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta charset="utf-8"/>
<!-- Google Tag Manager -->
<script type="text">(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
  new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
  j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
  'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
  })(window,document,'script','dataLayer','GTM-NHZSKTD');</script>
<!-- End Google Tag Manager -->
<script type="application/ld+json">
        {
            "@context": "https://schema.org",
            "@type": "Organization",
            "url": "https://www.gov.br/cgu",
            "logo": "https://www.gov.br/cgu/logo.png"
        }</script>
<script type="application/ld+json">
        {
            "@context": "https://schema.org",
            "@type": "WebSite",
            "url": "https://www.gov.br/cgu",
            "potentialA

In [6]:
def read_csv_cgu(link):
    r = requests.get(link)
    r.raise_for_status()

    content = r.content

    # Detecta ZIP
    if zipfile.is_zipfile(io.BytesIO(content)):
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            nome = z.namelist()[0]
            with z.open(nome) as f:
                df = pl.read_csv(
                    f,
                    separator=";",
                    encoding="latin1",
                    truncate_ragged_lines=True,
                    ignore_errors=True
                )
    else:
        df = pl.read_csv(
            io.BytesIO(content),
            separator=";",
            encoding="latin1",
            truncate_ragged_lines=True,
            ignore_errors=True
        )

    return df

In [ ]:
dados = {}
base = "https://www.gov.br"

for h3 in tqdm(soup.find_all("h3"), desc="Processando anos"):
    texto = h3.get_text(strip=True)

    if re.match(r"^\d{4}$", texto):
        ano = texto
        dados[ano] = {}

    
        ul = h3.find_next_sibling("ul")
        if ul:
            for a in ul.find_all("a", href=True):
                mes = a.get_text(strip=True)
                link = urljoin(base, a["href"])

                try:
                    df = (
                        read_csv_cgu(link)
                        .with_columns([
                            pl.lit(int(ano)).alias("ano"),
                            pl.lit(mes).alias("mes")
                        ])
                    )

                    dados[ano][mes] = df
                    print(f"✔ {ano}-{mes}")

                except Exception as e:
                    print(f"✖ Erro em {ano}-{mes}: {e}")

Processando anos:   0%|          | 0/9 [00:00<?, ?it/s]

✔ 2025-Janeiro
✔ 2025-Maio


Processando anos:  11%|█         | 1/9 [00:09<01:17,  9.65s/it]

✔ 2025-Setembro
✔ 2024-Setembro
✔ 2024-Maio


Processando anos:  22%|██▏       | 2/9 [00:21<01:15, 10.78s/it]

...

✔ 2022-Janeiro
✔ 2021-Setembro
✔ 2021-Maio


Processando anos:  67%|██████▋   | 6/9 [01:49<01:00, 20.26s/it]

✔ 2021-Janeiro
✔ 2020-Setembro
✔ 2020-Maio


Processando anos:  78%|███████▊  | 7/9 [02:19<00:46, 23.10s/it]

✔ 2020-Janeiro


In [16]:
# visualizar resultado
from pprint import pprint
pprint(dados)

{'2019': {'Janeiro': shape: (97_827, 25)
┌────────────┬─────┬────────┬────────────────────┬───┬───────┬────────────────────┬──────┬─────────┐
│ ï»¿7070352 ┆ M.J ┆ 200109 ┆ DEPARTAMENTO DE    ┆ … ┆ 30802 ┆ 30802_duplicated_0 ┆ ano  ┆ mes     │
│ ---        ┆ --- ┆ ---    ┆ POLICIA RODOVI…    ┆   ┆ ---   ┆ ---                ┆ ---  ┆ ---     │
│ i64        ┆ str ┆ i64    ┆ ---                ┆   ┆ i64   ┆ i64                ┆ i32  ┆ str     │
│            ┆     ┆        ┆ str                ┆   ┆       ┆                    ┆      ┆         │
╞════════════╪═════╪════════╪════════════════════╪═══╪═══════╪════════════════════╪══════╪═════════╡
│ 7070353    ┆ M.J ┆ 200109 ┆ DEPARTAMENTO DE    ┆ … ┆ 30802 ┆ 30802              ┆ 2019 ┆ Janeiro │
│            ┆     ┆        ┆ POLICIA RODOVI…    ┆   ┆       ┆                    ┆      ┆         │
│ 7070354    ┆ M.J ┆ 200109 ┆ DEPARTAMENTO DE    ┆ … ┆ 30802 ┆ 30802              ┆ 2019 ┆ Janeiro │
│            ┆     ┆        ┆ POLICIA RODOVI…    ┆

In [10]:
for header in soup.find_all(text=lambda t: t.strip().isdigit()):
    year = header.strip()
    # encontra elementos próximos que sejam links
    parent = header.parent
    links = parent.find_next_siblings("ul")
    for ul in links:
        for a in ul.find_all("a", href=True):
            month = a.get_text(strip=True)
            href  = a["href"]
            # às vezes o href é relativo → normaliza
            if href.startswith("/"):
                href = "https://www.gov.br" + href

            # baixar CSV
            r = requests.get(href)
            if r.status_code == 200:
                filename = f"{year}_{month}.csv"
                with open(filename, "wb") as f:
                    f.write(r.content)
                print(f"Baixado: {filename}")
            else:
                print(f"Falha ao baixar {href}")

print("Download concluído!")

C:\Users\amori\AppData\Local\Temp\ipykernel_26392\4100917005.py:1: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for header in soup.find_all(text=lambda t: t.strip().isdigit()):


Baixado: 2025_Janeiro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Janeiro.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Janeiro.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Janeiro.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Janeiro.csv
Baixado: 2025_Setembro.csv
Baixado: 2025_Maio.csv
Baixado: 2025_Janeiro.csv
Baixado: 2024_Setembro.csv
Baixado: 2024_Maio.csv
Baixado: 2024_Janeiro.csv
Baixado: 2024_Setembro.csv
Baixado: 2024_Maio.csv
Baixado: 2024_Janeiro.csv
Baixado: 2024_Setembro.csv
Baixado: 2024_Maio.csv
Baixado: 2024_Janeiro.csv


KeyboardInterrupt: 